In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from datetime import datetime
from typing import List, Dict, Any, Tuple

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, log_loss
from xgboost import XGBClassifier

In [2]:
# pipeline.py


# ---------- PARAMETERS ----------
# df_model must already contain:
# - one row per prediction opportunity
# - a column "SaleTransactionDate" of dtype datetime
# - target column "next_target_group" (string label of category/family)
# - any engineered feature columns (pref_*, cum_spent, days_since_last_purchase, ...)

df_model = pd.read_csv("../data/transformed/df_model_final_family_level1.csv")
TARGET_COL = "next_target_group"         # string label (not encoded)
DATE_COL = "SaleTransactionDate"
ID_COL = "ClientID"

MODEL_DIR = "../models"
os.makedirs(MODEL_DIR, exist_ok=True)

# Time split: use a date cutoff for test (adjust to your data)
TEST_CUTOFF = pd.Timestamp("2025-01-01", tz='UTC')   # example: all rows >= this => test
VALIDATION_WINDOW_DAYS = 90                # last 90 days before TEST_CUTOFF used for validation

RANDOM_STATE = 42
TOP_K = 5

In [3]:
df_model["SaleTransactionDate"].max()

'2025-02-14 00:00:00+00:00'

In [4]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_model["y"] = le.fit_transform(df_model["next_target_group"])

In [5]:
le.inverse_transform(df_model["y"]) # will be used to compare predictions in the end


array(['Ball', 'Jersey', 'Ball', ..., 'Ball', 'Ball', 'Shuttlecock'],
      shape=(867125,), dtype=object)

In [6]:
## For reference, what does "y" corresponds to :
df_model["next_target_group"]

0                Ball
1              Jersey
2                Ball
3                Ball
4               Shoes
             ...     
867120           Ball
867121           Ball
867122           Ball
867123           Ball
867124    Shuttlecock
Name: next_target_group, Length: 867125, dtype: object

In [7]:
exclude = [ID_COL, DATE_COL, TARGET_COL, "y", "Unnamed: 0.1", "Unnamed: 0"]
candidate = [c for c in df_model.columns if c not in exclude]
numeric_cols = df_model[candidate].select_dtypes(include=["number"]).columns.tolist()
categorical_cols = [c for c in candidate if c not in numeric_cols]

In [8]:
numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])
preprocessor = ColumnTransformer(
    [
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols),
    ],
    remainder="drop",
    sparse_threshold=0
)

In [9]:
date_col = DATE_COL
test_cutoff = TEST_CUTOFF
val_window_days = VALIDATION_WINDOW_DAYS


df = df_model.copy()
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
test_mask = df[date_col] >= test_cutoff
train_mask = df[date_col] < test_cutoff

train_df = df[train_mask].copy()
test_df = df[test_mask].copy()

In [10]:
X_train = preprocessor.fit_transform(train_df[numeric_cols + categorical_cols])
y_train = train_df["y"].values

X_test = preprocessor.transform(test_df[numeric_cols + categorical_cols]) if len(test_df) else None
y_test = test_df["y"].values if len(test_df) else None

In [11]:
model = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    n_estimators=300,        # 1000 -> 300
    learning_rate=0.1,       # 0.05 -> 0.1 (need fewer trees)
    max_depth=0,
    max_leaves=32,           # 64 -> 32
    grow_policy="lossguide",
    min_child_weight=10,     # 5 -> 10
    subsample=0.7,           # 0.8 -> 0.7
    colsample_bytree=0.7,    # 0.8 -> 0.7
    gamma=0.2,
    tree_method="hist",
    max_bin=128,             # 256 -> 128
    n_jobs=-1,
    random_state=RANDOM_STATE
)

model.fit(X_train, y_train)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.7
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [12]:
def top_k_accuracy(y_true: np.ndarray, y_proba: np.ndarray, k: int) -> float:
    topk = np.argsort(y_proba, axis=1)[:, -k:][:, ::-1]  # top k indices
    hits = 0
    for i, true in enumerate(y_true):
        if true in topk[i]:
            hits += 1
    return hits / len(y_true)

def mrr_score(y_true: np.ndarray, y_proba: np.ndarray) -> float:
    order = np.argsort(y_proba, axis=1)[:, ::-1]  # descending order
    rr_sum = 0.0
    n = len(y_true)
    for i, true in enumerate(y_true):
        ranks = np.where(order[i] == true)[0]
        if ranks.size > 0:
            rr_sum += 1.0 / (ranks[0] + 1.0)
    return rr_sum / n

In [13]:
joblib.dump(model, os.path.join(MODEL_DIR, "xgb_model_family_level1.joblib"))


['../models/xgb_model_family_level1.joblib']

In [14]:
def eval_and_print(X, y, split_name="set"):
        if X is None or len(y)==0:
            print(f"No data for {split_name}")
            return
        proba = model.predict_proba(X)
        acc = accuracy_score(y, np.argmax(proba, axis=1))
        # loss = log_loss(y, proba)
        top1 = top_k_accuracy(y, proba, 1)
        top5 = top_k_accuracy(y, proba, TOP_K)
        mrr = mrr_score(y, proba)
        print(f"== {split_name} metrics ==")
        print(f"Accuracy: {acc:.4f}; Top-1: {top1:.4f}; Top-{TOP_K}: {top5:.4f}; MRR: {mrr:.4f}")

eval_and_print(X_train, y_train, "train")
eval_and_print(X_test, y_test, "test")

    # Feature importances
try:
    importances = model.get_booster().get_score(importance_type="gain")
    sorted_imp = sorted(importances.items(), key=lambda x: x[1], reverse=True)[:30]
    print("Top feature importances (gain):")
    for feat, val in sorted_imp:
        print(feat, val)
except Exception as e:
    print("Could not extract feature importances:", e)

== train metrics ==
Accuracy: 0.4471; Top-1: 0.4471; Top-5: 0.9275; MRR: 0.6421
== test metrics ==
Accuracy: 0.4567; Top-1: 0.4567; Top-5: 0.9169; MRR: 0.6450
Top feature importances (gain):
f12 84.16439819335938
f18 78.74808502197266
f33 77.99076080322266
f19 72.4804916381836
f11 62.602500915527344
f29 59.069576263427734
f32 55.61336898803711
f26 53.3061408996582
f27 49.96323776245117
f20 49.50785446166992
f25 48.56864547729492
f28 47.709075927734375
f10 46.33488082885742
f16 41.05141067504883
f36 37.201576232910156
f21 36.76984786987305
f17 30.5273380279541
f45 28.58445930480957
f14 25.148433685302734
f37 24.791366577148438
f23 24.19448471069336
f13 20.457155227661133
f44 16.126367568969727
f7 14.803449630737305
f31 9.638620376586914
f34 9.440468788146973
f24 8.627861976623535
f41 6.779662609100342
f43 6.748926639556885
f6 5.968227863311768


In [15]:
def predict_top_n_for_client(
    df_row: pd.DataFrame,                 # one row of features for the client (pandas DataFrame with same columns as training features)
    preprocessor,
    model: XGBClassifier,
    label_encoder: LabelEncoder,
    top_n: int = 5
) -> List[Tuple[str, float]]:
    """
    Returns list of (label, prob) sorted by prob desc
    df_row should contain the feature columns (not target nor IDs), shape (1, n_features)
    """
    X_proc = preprocessor.transform(df_row)
    proba = model.predict_proba(X_proc)[0]   # shape (n_classes,)
    top_idx = np.argsort(proba)[-top_n:][::-1]
    labels = label_encoder.inverse_transform(top_idx)
    return list(zip(labels, proba[top_idx]))

In [16]:
def predict_top_k_names(model, X, le, prod_id_to_name, k=5):
    """
    Return a list-of-lists of top-k product NAMES for each row in X.
    """
    proba = model.predict_proba(X)                # shape (n_rows, n_classes)
    topk_idx = np.argsort(proba, axis=1)[:, ::-1][:, :k]  # indices (class ids) desc

    topk_names = []
    for row_idx in range(topk_idx.shape[0]):
        class_idxs = topk_idx[row_idx]
        # convert class index -> original product id (via LabelEncoder)
        prod_ids = le.inverse_transform(class_idxs)
        # convert product id -> product name (fallback to id string if missing)
        names = [prod_id_to_name.get(pid, str(pid)) for pid in prod_ids]
        topk_names.append(names)

    return topk_names

In [17]:
products = pd.read_csv("../data/raw/products.csv")

In [18]:
prod_id_to_name = dict(zip(products.ProductID.values, products.FamilyLevel2.values))

In [19]:
predict_top_k_names(model, X_test, le, prod_id_to_name, k=5)

[['Ball', 'Shoes', 'Shorts', 'Jersey', 'Racket'],
 ['Ball', 'Jersey', 'Shoes', 'Helmet', 'Bike'],
 ['Ball', 'Jersey', 'Shoes', 'Clubs', 'Bike'],
 ['Ball', 'Shoes', 'Jersey', 'Helmet', 'Bike'],
 ['Ball', 'Shoes', 'Jersey', 'Bike', 'Helmet'],
 ['Jersey', 'Stick', 'Ball', 'Shoes', 'Shorts'],
 ['Ball', 'Glove', 'Jersey', 'Shoes', 'Shorts'],
 ['Ball', 'Glove', 'Shoes', 'Shorts', 'Jersey'],
 ['Ball', 'Glove', 'Shoes', 'Bat', 'Racket'],
 ['Shoes', 'Ball', 'Jersey', 'Racket', 'Shorts'],
 ['Ball', 'Jersey', 'Shoes', 'Shorts', 'Racket'],
 ['Ball', 'Jersey', 'Shoes', 'Shorts', 'Racket'],
 ['Ball', 'Jersey', 'Shoes', 'Racket', 'Shorts'],
 ['Ball', 'Jersey', 'Shoes', 'Shorts', 'Racket'],
 ['Helmet', 'Stick', 'Puck', 'Ball', 'Shoes'],
 ['Ball', 'Racket', 'Shoes', 'Jersey', 'Shorts'],
 ['Ball', 'Racket', 'Shoes', 'Shorts', 'Jersey'],
 ['Ball', 'Bike', 'Jersey', 'Shoes', 'Helmet'],
 ['Ball', 'Jersey', 'Shoes', 'Shorts', 'Bike'],
 ['Ball', 'Jersey', 'Shoes', 'Bike', 'Shorts'],
 ['Bike', 'Ball', 'Shoes'

In [20]:
predict_top_n_for_client()

TypeError: predict_top_n_for_client() missing 4 required positional arguments: 'df_row', 'preprocessor', 'model', and 'label_encoder'

In [ ]:
def save_artifacts(preprocessor, label_encoder, model, out_dir=MODEL_DIR):
    joblib.dump(preprocessor, os.path.join(out_dir, "preprocessor.joblib"))
    joblib.dump(label_encoder, os.path.join(out_dir, "label_encoder.joblib"))
    joblib.dump(model, os.path.join(out_dir, "xgb_model.joblib"))

def load_artifacts(out_dir=MODEL_DIR):
    preprocessor = joblib.load(os.path.join(out_dir, "preprocessor.joblib"))
    label_encoder = joblib.load(os.path.join(out_dir, "label_encoder.joblib"))
    model = joblib.load(os.path.join(out_dir, "xgb_model.joblib"))
    return preprocessor, label_encoder, model